# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pathakadithi/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can historical search-performance signals be used to rank content
for review based on observed changes in subsequent search performance?

### Decision Supported

This analysis supports prioritizing pages for content review or
refresh investigation. The resulting score is a decision-support
ranking based on observed search-performance patterns; it does not
claim that refreshing a page will cause a specific change in search
performance.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data source

This analysis uses the public-safe FlyRank internship warehouse release
from the `FlyRank/internship-warehouse` dataset.

The main tables used are:

- `fact_content_query_90d` — historical content × query × 90-day-window
  search-performance records.
- `fact_content_daily_performance` — daily content-level performance
  records used to verify the development and outcome windows.

### Observation windows

March 2026 is used as the development and verification window.

June 2026 is kept sealed as the natural outcome/test period rather than
being used to construct the decision-time features.

The query-level table is aggregated to the content/page decision unit
before producing ranked recommendations.

### Exclusions

Future-period performance information is excluded from decision-time
features to reduce leakage. Fields that would only be known after the
decision point, including future-month performance measures, are not
used as model inputs.

The analysis also avoids client-identifying information, private
queries, domains, credentials, and raw exports in the public-facing
work.

### Data grain

The `fact_content_query_90d` table is at a content × query × 90-day-window
grain. A record is identified by content, query, and window fields.
The final decision unit for this lane is the content/page.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Decision unit

The final decision unit is a content/page. Query-level search
performance is aggregated to this level before scoring content.

### Features

The model uses historical search-performance signals available at the
decision point:

- impressions over the historical window
- clicks over the historical window
- click-through rate (CTR)
- average position over the historical window
- recent average position

These features describe observed historical performance and are intended
to support prioritization rather than establish causal relationships.

### Outcome / label

The outcome is defined using performance observed in a later sealed
window. The purpose is to identify content whose subsequent performance
shows an opportunity signal relative to its earlier performance.

The future outcome is kept separate from the features so that information
from the outcome period cannot influence the decision-time score.

### Baseline

The existing rule-based baseline combines two observed signals:

1. the gap between observed CTR and a position-based expected CTR;
2. recent CTR decline.

The two components are combined into a baseline opportunity score and
converted into an action/ranking queue.

### Validation design

The final evaluation separates the historical decision window from the
later outcome window. The model is evaluated on outcomes that were not
available when the features were constructed.

Model performance is compared with the rule-based baseline using the same
evaluation population.

### Leakage checks

Future-window performance variables are not used as model features.
Features are constructed only from information available before the
outcome window.

The validation design therefore tests whether historical signals provide
useful directional information about later observed performance rather
than measuring the model's ability to reproduce the baseline rules.

In [2]:
# Load the warehouse tables needed for the capstone

from datasets import load_dataset

# Load the query-level warehouse table
query_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

# Load the daily performance table
daily_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

print("Query table:", query_data)
print("Daily performance table:", daily_data)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Query table: DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share'],
        num_rows: 2414248
    })
})
Daily performance table: DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_

In [3]:
# Convert the query-level table to pandas
query_df = query_data["train"].to_pandas()

# Create CTR safely
query_df["ctr_90d"] = np.where(
    query_df["impressions_90d"] > 0,
    query_df["clicks_90d"] / query_df["impressions_90d"],
    0
)

# Create a recent-vs-previous CTR signal
query_df["ctr_last30"] = np.where(
    query_df["impressions_last30"] > 0,
    query_df["clicks_last30"] / query_df["impressions_last30"],
    0
)

query_df["ctr_prev30"] = np.where(
    query_df["impressions_prev30"] > 0,
    query_df["clicks_prev30"] / query_df["impressions_prev30"],
    0
)

# Historical decline proxy
query_df["ctr_decline"] = (
    query_df["ctr_last30"] - query_df["ctr_prev30"]
)

print("Rows:", len(query_df))
print("Columns:", query_df.columns.tolist())

Rows: 2414248
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share', 'ctr_90d', 'ctr_last30', 'ctr_prev30', 'ctr_decline']


In [4]:
# Aggregate query-level observations to the content/page decision unit

content_df = (
    query_df
    .groupby("content_hash_id", as_index=False)
    .agg(
        impressions_90d=("impressions_90d", "sum"),
        clicks_90d=("clicks_90d", "sum"),
        impressions_last30=("impressions_last30", "sum"),
        clicks_last30=("clicks_last30", "sum"),
        impressions_prev30=("impressions_prev30", "sum"),
        clicks_prev30=("clicks_prev30", "sum"),
        avg_position_90d=("avg_position_90d", "mean"),
        avg_position_last30=("avg_position_last30", "mean"),
        avg_position_prev30=("avg_position_prev30", "mean"),
        visible_query_count=("query_hash_id", "nunique"),
    )
)

# Recalculate content-level CTRs after aggregation
content_df["ctr_90d"] = np.where(
    content_df["impressions_90d"] > 0,
    content_df["clicks_90d"] / content_df["impressions_90d"],
    0
)

content_df["ctr_last30"] = np.where(
    content_df["impressions_last30"] > 0,
    content_df["clicks_last30"] / content_df["impressions_last30"],
    0
)

content_df["ctr_prev30"] = np.where(
    content_df["impressions_prev30"] > 0,
    content_df["clicks_prev30"] / content_df["impressions_prev30"],
    0
)

content_df["ctr_decline"] = (
    content_df["ctr_last30"] - content_df["ctr_prev30"]
)

print("Content/page rows:", len(content_df))
print("\nColumns:")
print(content_df.columns.tolist())

print("\nMissing values:")
print(content_df.isna().sum())

Content/page rows: 133852

Columns:
['content_hash_id', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'visible_query_count', 'ctr_90d', 'ctr_last30', 'ctr_prev30', 'ctr_decline']

Missing values:
content_hash_id            0
impressions_90d            0
clicks_90d                 0
impressions_last30         0
clicks_last30              0
impressions_prev30         0
clicks_prev30              0
avg_position_90d           0
avg_position_last30    16848
avg_position_prev30    13666
visible_query_count        0
ctr_90d                    0
ctr_last30                 0
ctr_prev30                 0
ctr_decline                0
dtype: int64


In [2]:
# Inspect daily-performance dates without converting all 78M rows to pandas

from datasets import load_dataset
import pandas as pd

# Reload only the dataset object — do NOT convert all rows to pandas
daily_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

daily_train = daily_data["train"]

print("Daily rows:", daily_train.num_rows)

print("\nColumns:")
print(daily_train.column_names)

# Only inspect a small sample
sample = daily_train.select(range(min(10000, daily_train.num_rows))).to_pandas()

sample["report_date"] = pd.to_datetime(sample["report_date"])

print("\nSample date range:")
print("Earliest:", sample["report_date"].min())
print("Latest:", sample["report_date"].max())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Daily rows: 78835655

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Sample date range:
Earliest: 2025-01-27 00:00:00
Latest: 2025-02-14 00:00:00


In [6]:
import pandas as pd

daily_train = daily_data["train"]

sample = daily_train.select(
    range(min(100000, daily_train.num_rows))
).to_pandas()

sample["report_date"] = pd.to_datetime(sample["report_date"])

print("Sample rows:", len(sample))
print("Sample earliest date:", sample["report_date"].min())
print("Sample latest date:", sample["report_date"].max())
print("Sample unique dates:", sample["report_date"].nunique())

Sample rows: 100000
Sample earliest date: 2025-01-27 00:00:00
Sample latest date: 2025-03-21 00:00:00
Sample unique dates: 41


In [7]:
# Check overlap between query-level content data and daily-performance sample

query_content_ids = set(content_df["content_hash_id"].dropna().unique())
daily_sample_content_ids = set(sample["content_hash_id"].dropna().unique())

overlap = query_content_ids.intersection(daily_sample_content_ids)

print("Unique content IDs in query/content table:", len(query_content_ids))
print("Unique content IDs in daily sample:", len(daily_sample_content_ids))
print("Overlapping content IDs:", len(overlap))

print(
    "Overlap percentage:",
    round(len(overlap) / len(query_content_ids) * 100, 2),
    "%"
)

NameError: name 'content_df' is not defined

In [8]:
print("daily_data:", "daily_data" in globals())
print("daily_train:", "daily_train" in globals())
print("sample:", "sample" in globals())
print("query_df:", "query_df" in globals())
print("content_df:", "content_df" in globals())

daily_data: True
daily_train: True
sample: True
query_df: False
content_df: False


In [9]:
# Check daily-performance content coverage using the existing sample

print("Daily sample rows:", len(sample))
print("Unique content IDs:", sample["content_hash_id"].nunique())
print("Unique clients:", sample["client_hash_id"].nunique())
print("Unique dates:", sample["report_date"].nunique())

print("\nDaily performance fields available:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
])

Daily sample rows: 100000
Unique content IDs: 7611
Unique clients: 4
Unique dates: 41

Daily performance fields available:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'sessions_organic', 'sessions_ai', 'scroll_events']


In [10]:
# Inspect how many rows we have for each date in the sample

daily_date_counts = (
    sample.groupby("report_date")
    .size()
    .reset_index(name="rows")
)

print(daily_date_counts.to_string(index=False))

report_date  rows
 2025-01-27   303
 2025-01-28   317
 2025-01-29   262
 2025-01-30   194
 2025-01-31   221
 2025-02-01   242
 2025-02-02   306
 2025-02-03   408
 2025-02-04   933
 2025-02-05  1087
 2025-02-06  1286
 2025-02-07  1381
 2025-02-08  1416
 2025-02-09  1489
 2025-02-10  1474
 2025-02-11  1703
 2025-02-12  2582
 2025-02-13  2751
 2025-02-14  2809
 2025-02-15  2995
 2025-02-16  3038
 2025-02-17  3431
 2025-02-18  4101
 2025-02-19  4221
 2025-02-20  4459
 2025-02-21  4398
 2025-02-22  4243
 2025-02-23  4290
 2025-02-24  4204
 2025-02-25  4238
 2025-02-26  4251
 2025-02-27  4203
 2025-02-28  4046
 2025-03-01  1359
 2025-03-02  4270
 2025-03-03  4371
 2025-03-15   207
 2025-03-16  2511
 2025-03-19   306
 2025-03-20  4887
 2025-03-21  4807


In [13]:
from datetime import date

daily_recent = daily_train.filter(
    lambda x: x["report_date"] >= date(2025, 6, 1)
)

print("Filtered rows:", daily_recent.num_rows)


Filter:   0%|          | 0/78835655 [00:00<?, ? examples/s]

KeyboardInterrupt: 

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
